# Statistics

Questo notebook legge `results/results.json`, verifica la coerenza delle checksum e genera i plot dei tempi di esecuzione.


In [ ]:
from collections import defaultdict
from pathlib import Path
import json

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
    PLOT_IMPORT_ERROR = ""
except Exception as exc:
    plt = None
    MATPLOTLIB_AVAILABLE = False
    PLOT_IMPORT_ERROR = str(exc)

if MATPLOTLIB_AVAILABLE:
    try:
        from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
        HAS_3D = True
    except Exception:
        HAS_3D = False
else:
    HAS_3D = False

RESULTS_PATH = Path("results/results.json")


def is_metrics(node):
    return (
        isinstance(node, dict)
        and {"throughput", "time", "checksum", "hash"} <= set(node.keys())
    )


def parse_results(payload):
    rows = []

    for top_key, level1 in payload.items():
        try:
            n_str, p_str = top_key.split()
        except ValueError as exc:
            raise ValueError(f"Invalid top-level key: {top_key!r}") from exc

        N = int(n_str)
        P = int(p_str)

        for key1, value1 in level1.items():
            if is_metrics(value1):
                rows.append(
                    {
                        "N": N,
                        "P": P,
                        "hash": value1.get("hash", ""),
                        "executable": key1,
                        "throughput": float(value1["throughput"]),
                        "time": float(value1["time"]),
                        "checksum": value1["checksum"],
                    }
                )
                continue

            if not isinstance(value1, dict):
                raise ValueError(f"Unsupported JSON layout under {top_key!r}")

            for key2, value2 in value1.items():
                if is_metrics(value2):
                    rows.append(
                        {
                            "N": N,
                            "P": P,
                            "hash": key1,
                            "executable": key2,
                            "throughput": float(value2["throughput"]),
                            "time": float(value2["time"]),
                            "checksum": value2["checksum"],
                        }
                    )
                    continue

                if not isinstance(value2, dict):
                    raise ValueError(f"Unsupported JSON layout under {top_key!r}/{key1!r}")

                for key3, value3 in value2.items():
                    if not is_metrics(value3):
                        raise ValueError(
                            f"Unsupported JSON layout under {top_key!r}/{key1!r}/{key2!r}"
                        )

                    rows.append(
                        {
                            "N": N,
                            "P": P,
                            "hash": key2,
                            "executable": key1,
                            "throughput": float(value3["throughput"]),
                            "time": float(value3["time"]),
                            "checksum": value3["checksum"],
                        }
                    )

    return rows


if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Results file not found: {RESULTS_PATH}. Run the benchmarks first."
    )

with RESULTS_PATH.open() as fh:
    payload = json.load(fh)

rows = parse_results(payload)
if not rows:
    raise ValueError("results/results.json exists but does not contain benchmark rows")

executables = sorted({row["executable"] for row in rows})
hashes = sorted({row["hash"] for row in rows})
n_values = sorted({row["N"] for row in rows})
p_values = sorted({row["P"] for row in rows})

print(f"Loaded {len(rows)} runs from {RESULTS_PATH}.")
print(f"Executables: {executables}")
print(f"Hashes: {hashes}")
print(f"N values: {n_values}")
print(f"P values: {p_values}")


In [ ]:
checksum_groups = defaultdict(set)

for row in rows:
    checksum_groups[(row["N"], row["P"], row["hash"])].add(row["checksum"])

inconsistent = {
    key: values for key, values in checksum_groups.items() if len(values) > 1
}

if inconsistent:
    print("Checksum mismatch found:")
    for (N, P, hash_name), values in sorted(inconsistent.items()):
        print(f"  N={N}, P={P}, hash={hash_name}: {sorted(values)}")
else:
    print("Checksums are coherent across implementations for each (N, P, hash).")


In [ ]:
if not MATPLOTLIB_AVAILABLE:
    print(f"Plotting skipped: matplotlib is not available ({PLOT_IMPORT_ERROR}).")
else:
    color_map = plt.get_cmap("tab10")

if MATPLOTLIB_AVAILABLE and len(n_values) > 1 and len(p_values) > 1 and HAS_3D:
    for hash_name in hashes:
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection="3d")

        for idx, executable in enumerate(executables):
            subset = [
                row
                for row in rows
                if row["hash"] == hash_name and row["executable"] == executable
            ]
            if not subset:
                continue

            subset.sort(key=lambda row: (row["N"], row["P"]))
            ax.scatter(
                [row["N"] for row in subset],
                [row["P"] for row in subset],
                [row["time"] for row in subset],
                label=executable,
                color=color_map(idx % 10),
                s=60,
            )

        ax.set_title(f"Runtime by N and P ({hash_name})")
        ax.set_xlabel("N")
        ax.set_ylabel("P")
        ax.set_zlabel("time [s]")
        ax.legend()
        plt.show()
elif MATPLOTLIB_AVAILABLE:
    for hash_name in hashes:
        for N in n_values:
            fig, ax = plt.subplots(figsize=(9, 5))
            plotted = False

            for idx, executable in enumerate(executables):
                subset = [
                    row
                    for row in rows
                    if row["hash"] == hash_name
                    and row["N"] == N
                    and row["executable"] == executable
                ]
                if not subset:
                    continue

                subset.sort(key=lambda row: row["P"])
                ax.plot(
                    [row["P"] for row in subset],
                    [row["time"] for row in subset],
                    marker="o",
                    label=executable,
                    color=color_map(idx % 10),
                )
                plotted = True

            if not plotted:
                plt.close(fig)
                continue

            ax.set_title(f"Runtime vs P for N={N} ({hash_name})")
            ax.set_xlabel("P")
            ax.set_ylabel("time [s]")
            ax.set_xticks(sorted({row["P"] for row in rows if row["N"] == N}))
            ax.grid(True, linestyle="--", alpha=0.4)
            ax.legend()
            plt.show()
